# V28 - Non-Building Subgraph GNN

## v25 → v28 변경사항
- **v25**: 10,000노드 full grid → GT edge 비율 0.025% → class imbalance
- **v28**: 건물 제외 free pixel만 노드 → ~5,000노드, GT edge 비율 ~0.5% (20배 개선)

## GT 파이프라인
- 4방향 gate × 4 cost function(Dijkstra×3 + A*) = 16 variant → pixel vote → consensus
- 49개 캠퍼스 수동 선택 → `collegemap/gt_masks_final/` (c4/c8/c12 per campus)
- 학습 시 GT 증강 없음 (campus당 1개 고정 GT)

## v25에서 그대로 유지
- gate_nodes + cluster_nodes → terminals 구조
- AMP safe 학습 루프 (gradient accumulation)
- BCE + Dice + FP ridge 손실
- Train/Val/Test 분리 (49 osm = train+val, 나머지 = test)

In [ ]:
import os
!git -C swe3032 pull 2>/dev/null || git clone https://github.com/SKKUAIProjectTeam1/swe3032.git
os.chdir('swe3032')

In [ ]:
import glob, os, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy.ndimage import distance_transform_edt, maximum_filter, gaussian_filter

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
CODE_VERSION  = 'v28a_subgraph_mlp'
print(f'코드 버전: {CODE_VERSION}')
RES           = 100
N             = RES * RES
NODE_DIM      = 9   # ridge, dist_n, x, y, cluster_indicator, dist_to_cluster, dist_to_gate, dy_cluster, dx_cluster
GAT_HEADS     = 4
EPOCHS        = 220
LR            = 4e-4
WARMUP_EPOCHS = 20
ATTN_DROPOUT  = 0.05
ACCUM_STEPS   = 4
N_GATES       = 2
GATE_MIN_DIST = 14
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLUSTER_EPS          = 13.0
CLUSTER_SPLIT_RADIUS = 15.0   # 클러스터 반경 초과 시 sub-cluster로 분할
ROAD_CLEAR_MIN    = 3.0
ROAD_CLEAR_MAX    = 24.0
RIDGE_FILTER_SIZE = 7
EDGE_THR          = 0.50
PRUNE_ITERS       = 12

IMG_DIR      = 'collegemap/images'
TXT_DIR      = 'collegemap/txt'
GT_FINAL_DIR = 'collegemap/gt_masks_final'
OUT_DIR      = 'output'
CKPT         = os.path.join(OUT_DIR, f'{CODE_VERSION}.pth')
CKPT_BEST    = os.path.join(OUT_DIR, f'{CODE_VERSION}_best.pth')
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── 공통 헬퍼 (v25와 동일) ──────────────────────────────────────────────────────
def _nearest_free_node(cy, cx, is_bld_grid):
    non_bld_yx = np.argwhere(is_bld_grid == 0)
    if len(non_bld_yx) == 0: return int(cy)*RES + int(cx)
    dists = (non_bld_yx[:,0]-cy)**2 + (non_bld_yx[:,1]-cx)**2
    best  = non_bld_yx[np.argmin(dists)]
    return int(best[0])*RES + int(best[1])

def _make_common_road_ridge(is_bld_grid):
    free  = is_bld_grid == 0
    dist  = distance_transform_edt(free)
    clear = (dist > ROAD_CLEAR_MIN) & (dist < ROAD_CLEAR_MAX)
    campus_influence = gaussian_filter(is_bld_grid.astype(np.float32), sigma=7.0)
    near_campus = campus_influence > 0.01
    local_max   = dist == maximum_filter(dist, size=RIDGE_FILTER_SIZE)
    ridge       = free & clear & near_campus & local_max
    ridge_score = gaussian_filter(ridge.astype(np.float32), sigma=1.2)
    if ridge_score.max() < 1e-6:
        band = free & clear & near_campus
        ridge_score = gaussian_filter(band.astype(np.float32), sigma=1.0)
    ridge_score = ridge_score / (ridge_score.max() + 1e-6)
    return ridge_score.astype(np.float32), dist.astype(np.float32)

def _cluster_centers(centers, eps=CLUSTER_EPS):
    """Ball clustering: 씨앗 기준 반경 eps 이내만 포함 (single-linkage 체인 방지)."""
    n=len(centers); assigned=[False]*n; clusters=[]
    for i in range(n):
        if assigned[i]: continue
        cy,cx=centers[i]; assigned[i]=True; cluster=[i]
        for j in range(n):
            if not assigned[j]:
                vy,vx=centers[j]
                if ((vy-cy)**2+(vx-cx)**2)**0.5 <= eps:
                    assigned[j]=True; cluster.append(j)
        clusters.append(cluster)
    return clusters

def _nearest_ridge_node(cy, cx, is_bld_grid, ridge_grid):
    candidates = np.argwhere((is_bld_grid==0) & (ridge_grid>0.05))
    if len(candidates) == 0:
        return _nearest_free_node(cy, cx, is_bld_grid)
    dists = (candidates[:,0]-cy)**2 + (candidates[:,1]-cx)**2
    best  = candidates[np.argmin(dists)]
    return int(best[0])*RES + int(best[1])

# v25와 동일: 경계 진입점(gate) 선택
_BOUNDARY = sorted(set(
    [i for i in range(10, RES-10)] +
    [(RES-1)*RES+i for i in range(10, RES-10)] +
    [i*RES for i in range(10, RES-10)] +
    [i*RES+(RES-1) for i in range(10, RES-10)]
))

def _select_gate_nodes(cluster_nodes, is_bld_grid, ridge_grid):
    """v25 그대로: 캠퍼스 중심 방향 경계 비건물 노드 N_GATES개 선택."""
    if len(cluster_nodes) == 0: return []
    cy = np.mean([n//RES for n in cluster_nodes])
    cx = np.mean([n%RES  for n in cluster_nodes])
    candidates = []
    for node in _BOUNDARY:
        y, x = divmod(int(node), RES)
        if is_bld_grid[y, x] > 0: continue
        score = ((y-cy)**2+(x-cx)**2)**0.5 - ridge_grid[y,x]*8.0
        candidates.append((score, int(node)))
    candidates.sort(key=lambda t: t[0])
    chosen = []
    for _, node in candidates:
        y, x = divmod(node, RES)
        if all(abs(x-(p%RES))+abs(y-(p//RES)) >= GATE_MIN_DIST for p in chosen):
            chosen.append(node)
        if len(chosen) == N_GATES: break
    return chosen

In [ ]:
# ── 서브그래프 구성 (v28 신규) ──────────────────────────────────────────────────
def build_subgraph(is_bld_grid):
    """건물 픽셀을 제외한 free pixel만으로 구성된 그래프.

    Returns:
        free_pixels   (n_free,)  픽셀 인덱스 [0..N-1]
        pixel_to_node (N,)       건물=-1, free=서브그래프 노드번호
        sub_ei        (2, E)     서브그래프 edge_index (torch.long, CPU)
    """
    is_bld_flat = is_bld_grid.flatten()
    free_pixels = np.where(is_bld_flat == 0)[0].astype(np.int32)
    n_free = len(free_pixels)

    pixel_to_node = np.full(N, -1, dtype=np.int32)
    pixel_to_node[free_pixels] = np.arange(n_free, dtype=np.int32)

    rows, cols = [], []
    for px in free_pixels:
        y, x = divmod(int(px), RES)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dy == 0 and dx == 0: continue
                ny, nx = y+dy, x+dx
                if 0 <= ny < RES and 0 <= nx < RES:
                    nb_node = pixel_to_node[ny*RES+nx]
                    if nb_node >= 0:
                        rows.append(int(pixel_to_node[px]))
                        cols.append(int(nb_node))

    return free_pixels, pixel_to_node, torch.tensor([rows, cols], dtype=torch.long)


def _make_sub_gt_edge_mask(gt_pixel_arr, free_pixels, sub_ei):
    """bool 100×100 배열 → 서브그래프 edge mask."""
    gt_flat = gt_pixel_arr.flatten()
    src_px  = free_pixels[sub_ei[0].numpy()]
    dst_px  = free_pixels[sub_ei[1].numpy()]
    return torch.tensor(gt_flat[src_px] & gt_flat[dst_px],
                        dtype=torch.float32, device=DEVICE)


def _direction_to_targets(free_pixels, target_nodes):
    """각 free pixel → 가장 가까운 target까지의 unit vector (dy, dx).
    
    클러스터가 먼 경우에도 모델이 '어느 방향으로 가야 하는지' 알 수 있게 한다.
    """
    n_free = len(free_pixels)
    if len(target_nodes) == 0:
        return np.zeros(n_free, np.float32), np.zeros(n_free, np.float32)
    py = (free_pixels // RES).astype(np.float32)
    px = (free_pixels  % RES).astype(np.float32)
    min_dy = np.zeros(n_free, np.float32)
    min_dx = np.zeros(n_free, np.float32)
    min_d2 = np.full(n_free, np.inf, np.float32)
    for t in target_nodes:
        ty, tx = divmod(int(t), RES)
        dy = ty - py; dx = tx - px
        d2 = dy*dy + dx*dx
        closer = d2 < min_d2
        min_d2[closer] = d2[closer]
        min_dy[closer] = dy[closer]
        min_dx[closer] = dx[closer]
    dist = np.sqrt(min_d2) + 1e-6
    return (min_dy / dist).astype(np.float32), (min_dx / dist).astype(np.float32)

In [ ]:
# ── 데이터 로딩 ─────────────────────────────────────────────────────────────────
def _find_txt(img_path):
    stem = os.path.basename(img_path).replace('_building_mask.png','')
    direct = os.path.join(TXT_DIR, stem+'_building_places.txt')
    if os.path.exists(direct): return direct
    prefix = stem.replace('-','_').split('_')[0]
    for fn in os.listdir(TXT_DIR):
        if fn.endswith('_building_places.txt') and fn.startswith(prefix):
            return os.path.join(TXT_DIR, fn)
    return None

def _dist_to_px(query_pixels, target_pixels, tau):
    if len(target_pixels) == 0:
        return np.zeros(len(query_pixels), dtype=np.float32)
    qy = query_pixels // RES; qx = query_pixels % RES
    feats = []
    for t in target_pixels:
        ty, tx = divmod(int(t), RES)
        feats.append(np.sqrt((qy-ty)**2 + (qx-tx)**2))
    dmin = np.stack(feats, axis=0).min(axis=0)
    return np.exp(-dmin / tau).astype(np.float32)

def _apply_flip(arr2d, flip):
    if   flip == 'lr':   return np.fliplr(arr2d).copy()
    elif flip == 'ud':   return np.flipud(arr2d).copy()
    elif flip == 'both': return np.fliplr(np.flipud(arr2d)).copy()
    return arr2d

def _flip_center(cy, cx, flip):
    if flip in ('lr', 'both'): cx = RES - cx
    if flip in ('ud', 'both'): cy = RES - cy
    return cy, cx

def _flip_poly(poly, W, H, flip):
    if flip is None: return poly
    out = {}
    for bid, pts in poly.items():
        if   flip == 'lr':   out[bid] = [(W - p[0], p[1])     for p in pts]
        elif flip == 'ud':   out[bid] = [(p[0],     H - p[1]) for p in pts]
        elif flip == 'both': out[bid] = [(W - p[0], H - p[1]) for p in pts]
    return out


def load_campus(img_path, txt_path, flip=None):
    img    = Image.open(img_path).convert('L')
    W, H   = img.size
    is_bld_raw = (np.array(img.resize((RES,RES),resample=Image.NEAREST)) > 128).astype(np.float32)
    is_bld = _apply_flip(is_bld_raw, flip)

    ridge, dist = _make_common_road_ridge(is_bld)
    dist_n = (dist / (dist.max()+1e-6)).astype(np.float32)

    ns={}; exec(open(txt_path, encoding='utf-8').read(), ns)
    poly_raw = ns['BUILDING_POLY']; building_ids = list(poly_raw.keys())
    poly = _flip_poly(poly_raw, W, H, flip)

    centers = []
    for bid in building_ids:
        pts = poly_raw[bid]
        cy  = np.mean([p[1] for p in pts]) * RES / H
        cx  = np.mean([p[0] for p in pts]) * RES / W
        cy, cx = _flip_center(cy, cx, flip)
        centers.append((cy, cx))

    bld_nodes = [_nearest_free_node(cy, cx, is_bld) for cy, cx in centers]
    clusters  = _cluster_centers(centers)
    cluster_nodes, cluster_groups = [], []
    for cl in clusters:
        cl_pts = [(centers[i][0], centers[i][1]) for i in cl]
        cy_m   = np.mean([p[0] for p in cl_pts])
        cx_m   = np.mean([p[1] for p in cl_pts])
        max_r  = max(((p[0]-cy_m)**2+(p[1]-cx_m)**2)**0.5 for p in cl_pts) if len(cl_pts)>1 else 0
        if max_r <= CLUSTER_SPLIT_RADIUS:
            final_groups = [cl]
        else:
            sub_cls      = _cluster_centers(cl_pts, eps=CLUSTER_SPLIT_RADIUS)
            final_groups = [[cl[j] for j in sub] for sub in sub_cls]
        for grp in final_groups:
            cy = np.mean([centers[i][0] for i in grp])
            cx = np.mean([centers[i][1] for i in grp])
            node = _nearest_ridge_node(cy, cx, is_bld, ridge)
            if node not in cluster_nodes:
                cluster_nodes.append(node)
                cluster_groups.append([building_ids[i] for i in grp])
    if len(cluster_nodes) <= 1 and len(bld_nodes) > 1:
        cluster_nodes  = list(dict.fromkeys(bld_nodes))
        cluster_groups = [[bid] for bid in building_ids]

    gate_nodes = _select_gate_nodes(cluster_nodes, is_bld, ridge)
    terminals  = list(dict.fromkeys(gate_nodes + cluster_nodes))

    free_pixels, pixel_to_node, sub_ei = build_subgraph(is_bld)
    n_free = len(free_pixels)

    def px_to_sub(px_list):
        return [int(pixel_to_node[p]) for p in px_list if pixel_to_node[p] >= 0]

    cluster_sub  = px_to_sub(cluster_nodes)
    gate_sub     = px_to_sub(gate_nodes)
    terminal_sub = px_to_sub(terminals)

    # ── GT 로드 + flip ────────────────────────────────────────────────────────
    slug    = os.path.basename(img_path).replace('_building_mask.png','')
    gt_path = os.path.join(GT_FINAL_DIR, f'{slug}_gt.npz')

    if os.path.exists(gt_path):
        gt_arr       = _apply_flip(np.load(gt_path)['gt'], flip)
        gt_edge_mask = _make_sub_gt_edge_mask(gt_arr, free_pixels, sub_ei)
        gt_source    = 'osm'
    else:
        gt_edge_mask = None
        gt_source    = 'test_no_osm'

    # ── 노드 피처 (NODE_DIM=9) ────────────────────────────────────────────────
    xs = (free_pixels % RES).astype(np.float32) / RES
    ys = (free_pixels // RES).astype(np.float32) / RES

    ridge_sub  = ridge.flatten()[free_pixels]
    dist_n_sub = dist_n.flatten()[free_pixels]

    cluster_indicator = np.zeros(n_free, dtype=np.float32)
    for nd in cluster_sub:
        if 0 <= nd < n_free: cluster_indicator[nd] = 1.0

    d_cluster            = _dist_to_px(free_pixels, cluster_nodes, tau=12.0)
    d_gate               = _dist_to_px(free_pixels, gate_nodes,    tau=15.0)
    dy_cluster, dx_cluster = _direction_to_targets(free_pixels, cluster_nodes)

    node_feats = np.stack([ridge_sub, dist_n_sub, xs, ys,
                           cluster_indicator, d_cluster, d_gate,
                           dy_cluster, dx_cluster], axis=1)  # (n_free, 9)

    src_n, dst_n = sub_ei[0].numpy(), sub_ei[1].numpy()
    edge_ridge   = torch.tensor((ridge_sub[src_n]+ridge_sub[dst_n])*0.5,
                                dtype=torch.float32, device=DEVICE)

    name = slug + (f'_{flip}' if flip else '')
    return {
        'node_feats':    torch.tensor(node_feats, dtype=torch.float32, device=DEVICE),
        'sub_ei':        sub_ei.to(DEVICE),
        'edge_ridge':    edge_ridge,
        'free_pixels':   free_pixels,
        'pixel_to_node': pixel_to_node,
        'n_free':        n_free,
        'is_bld':        is_bld,
        'ridge':         ridge,
        'cluster_nodes': cluster_nodes,
        'cluster_sub':   cluster_sub,
        'gate_nodes':    gate_nodes,
        'gate_sub':      gate_sub,
        'terminal_nodes':terminals,
        'terminal_sub':  terminal_sub,
        'cluster_groups':cluster_groups,
        'gt_edge_mask':  gt_edge_mask,
        'gt_source':     gt_source,
        'slug':          slug,
        'poly': poly, 'name': name, 'W': W, 'H': H,
    }


# GT 있는 캠퍼스 4방향 flip 증강, test는 원본만
campuses = []
for img_path in sorted(glob.glob(os.path.join(IMG_DIR,'*_building_mask.png'))):
    txt = _find_txt(img_path)
    if not txt: continue
    slug    = os.path.basename(img_path).replace('_building_mask.png','')
    has_gt  = os.path.exists(os.path.join(GT_FINAL_DIR, f'{slug}_gt.npz'))
    flips   = [None, 'lr', 'ud', 'both'] if has_gt else [None]
    for flip in flips:
        try:
            campuses.append(load_campus(img_path, txt, flip=flip))
        except Exception as e:
            print(f'skip {slug} flip={flip}: {e}')

# val/train 분리 — 원본 slug 기준으로 split, augmented는 train에만
osm_orig      = [c for c in campuses if c['gt_source']=='osm' and c['name']==c['slug']]
test_campuses = [c for c in campuses if c['gt_source']=='test_no_osm']

random.seed(42); random.shuffle(osm_orig)
n_val     = max(1, len(osm_orig)//7)
val_slugs = {c['slug'] for c in osm_orig[:n_val]}

val_campuses   = [c for c in osm_orig   if c['slug'] in val_slugs]
train_campuses = [c for c in campuses   if c['gt_source']=='osm' and c['slug'] not in val_slugs]

print(f'전체 로드: {len(campuses)}개')
print(f'Train: {len(train_campuses)} (4× aug)  Val: {len(val_campuses)} (원본)  Test: {len(test_campuses)}')
for c in train_campuses[:3]:
    ge = int(c['gt_edge_mask'].sum()); ratio = ge/c['sub_ei'].shape[1]*100
    print(f"  {c['name']}: n_free={c['n_free']}, gt_edges={ge} ({ratio:.2f}%)")

In [ ]:
# ── GT 시각화 (gt_masks_final 로드 확인) ────────────────────────────────────────
def visualize_gt(campus_name='aalto_university'):
    c = next((x for x in campuses if campus_name in x['name']), None)
    if c is None: print(f'{campus_name} not found'); return
    if c['gt_edge_mask'] is None: print(f'{campus_name}: GT 없음 (test)'); return

    W, H = c['W'], c['H']
    fp   = c['free_pixels']
    ei   = c['sub_ei'].cpu().numpy()
    gt   = c['gt_edge_mask'].cpu().numpy().astype(bool)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')

    for bid, pts in c['poly'].items():
        sc = [(p[0]*RES/W, p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc, closed=True,
                     facecolor='#2d3436', edgecolor='#00cec9', lw=0.8, alpha=0.9))

    for i, keep in enumerate(gt):
        if not keep: continue
        s_px = fp[ei[0,i]]; d_px = fp[ei[1,i]]
        y1,x1 = divmod(s_px,RES); y2,x2 = divmod(d_px,RES)
        ax.plot([x1,x2],[y1,y2], color='#ff6b6b', alpha=0.7, lw=1.5, zorder=3)

    for node in c['cluster_nodes']:
        y,x = divmod(node,RES)
        ax.scatter(x,y,s=80,c='#74b9ff',edgecolors='white',lw=0.7,zorder=9)
    for node in c['gate_nodes']:
        y,x = divmod(node,RES)
        ax.scatter(x,y,s=100,c='#fdcb6e',marker='D',edgecolors='white',lw=0.7,zorder=9)

    ge = int(gt.sum()); ratio = ge/len(gt)*100
    ax.set_xlim(0,RES); ax.set_ylim(RES,0); ax.axis('off')
    ax.set_title(f'{c["name"].replace("_"," ").title()}\n'
                 f'n_free={c["n_free"]}, gt_edges={ge} ({ratio:.2f}%)',
                 color='white', fontsize=11)
    plt.tight_layout(); plt.show(); plt.close()

print(f'train+val GT 캠퍼스: {len(osm_orig)}개 (원본), train={len(train_campuses)}, val={len(val_campuses)}')
for c in osm_orig[:3]:
    visualize_gt(c['name'])

In [ ]:
# ── 모델: 서브그래프 MLP (GNN/GAT 레이어 배제, 노드별 피처 변환 후 에지 예측) ───────────
class SubgraphMLP(nn.Module):
    """v28a_subgraph_gnn 구조에서 GNN(GAT) 레이어를 MLP 레이어로 대체하여
    이웃 정보 노드 전파 없이 각 노드/에지 피처만으로 도로망을 예측하는 모델.
    """
    def __init__(self):
        super().__init__()
        self.node_mlp = nn.Sequential(
            nn.Linear(NODE_DIM, 64),
            nn.LayerNorm(64),
            nn.ELU(),
            nn.Dropout(ATTN_DROPOUT),
            nn.Linear(64, 64),
            nn.LayerNorm(64),
            nn.ELU(),
            nn.Dropout(ATTN_DROPOUT),
            nn.Linear(64, 64),
            nn.LayerNorm(64),
            nn.ELU()
        )
        # v25와 동일: h_s, h_d, |h_s-h_d|, h_s*h_d + raw 2개
        self.edge_head = nn.Sequential(
            nn.Linear(64*4+2, 128), nn.ReLU(), nn.Dropout(0.08),
            nn.Linear(128, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
        nn.init.constant_(self.edge_head[-1].bias, -3.0)

    def forward(self, feats, ei):
        h3 = self.node_mlp(feats)
        src, dst = ei[0], ei[1]
        hs, hd = h3[src], h3[dst]
        raw_edge = torch.stack([
            (feats[src, 0] + feats[dst, 0]) * 0.5,  # ridge
            (src != dst).float(),               # self-loop 여부 (서브그래프엔 없지만 호환)
        ], dim=1)
        edge_feat = torch.cat([hs, hd, torch.abs(hs - hd), hs * hd, raw_edge], dim=1)
        edge_logits = self.edge_head(edge_feat).squeeze(-1).float()
        edge_logits = torch.nan_to_num(edge_logits, nan=0., posinf=20., neginf=-20.).clamp(-20., 20.)
        ew = torch.sigmoid(edge_logits)
        return ew, edge_logits


model = SubgraphMLP().to(DEVICE)
print(f'파라미터: {sum(p.numel() for p in model.parameters()):,}개')

c0 = campuses[0]
with torch.no_grad():
    ew_t, _ = model(c0['node_feats'], c0['sub_ei'])
print(f'forward ok: n_free={c0["n_free"]}, sub_edges={c0["sub_ei"].shape[1]}, ew={ew_t.shape}')

In [ ]:
# ── 손실 함수 ──────────────────────────────────────────────────────────────────
def trunk_bce_loss(edge_logits, gt_edge_mask):
    logits = torch.nan_to_num(edge_logits.float(),nan=0.,posinf=20.,neginf=-20.).clamp(-20.,20.)
    target = gt_edge_mask.float()
    pos = target.sum()
    if pos <= 0:
        return F.binary_cross_entropy_with_logits(logits, target)
    neg = target.numel() - pos
    pos_weight = torch.clamp(neg/(pos+1e-6), min=1., max=80.).detach()
    return F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)

def soft_dice_loss(ew, gt_edge_mask):
    pred   = ew.float().clamp(0.,1.)
    target = gt_edge_mask.float()
    inter  = (pred*target).sum()
    return 1.0 - (2.*inter+1.)/(pred.sum()+target.sum()+1.)

def false_positive_ridge_loss(ew, gt_edge_mask, edge_ridge):
    fp = gt_edge_mask < 0.5
    if fp.sum() == 0: return ew.sum()*0.
    pred = ew[fp].float().clamp(0.,1.)
    return (pred*(1.-edge_ridge[fp]).clamp(0.2,1.)).mean()

def cluster_connectivity_loss(ew, c):
    """각 terminal(gate+cluster) 노드에 weight > EDGE_THR인 엣지가 없으면 penalty.

    cluster_sub / gate_sub 노드 중 하나라도 고립되면 패널티를 줘서
    모델이 반드시 terminal 쪽으로 경로를 만들도록 유도한다.
    """
    ei        = c['sub_ei']          # (2, E) on DEVICE
    terminals = c['terminal_sub']    # list of ints
    if not terminals:
        return ew.new_zeros(())
    penalty = ew.new_zeros(())
    for t in terminals:
        incident = (ei[0] == t) | (ei[1] == t)
        if incident.any():
            penalty = penalty + F.relu(EDGE_THR - ew[incident].max())
    return penalty / len(terminals)

def loss_terms(ew, edge_logits, c, scale):
    bce   = trunk_bce_loss(edge_logits, c['gt_edge_mask'])
    dice  = soft_dice_loss(ew, c['gt_edge_mask'])
    fp    = false_positive_ridge_loss(ew, c['gt_edge_mask'], c['edge_ridge']) * (0.2+0.8*scale)
    conn  = cluster_connectivity_loss(ew, c) * (0.1+0.9*scale)
    sparse = ew.float().mean() * 0.03

    with torch.no_grad():
        pred = ew > EDGE_THR
        gt   = c['gt_edge_mask'] > 0
        tp   = (pred & gt).sum().float()
        precision = tp / (pred.sum().float()+1e-6)
        recall    = tp / (gt.sum().float()+1e-6)

    return {
        'bce':bce,'dice':dice,'fp':fp,'conn':conn,'sparse':sparse,
        'precision':precision.detach(),'recall':recall.detach(),
    }

def loss_fn(ew, edge_logits, c, scale):
    t = loss_terms(ew, edge_logits, c, scale)
    total = t['bce'] + 0.8*t['dice'] + t['fp'] + 0.5*t['conn'] + t['sparse']
    return torch.nan_to_num(total, nan=0., posinf=100., neginf=0.)

In [ ]:
# ── 학습 ────────────────────────────────────────────────────────────────────────
opt    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=5e-6)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
n = len(train_campuses); loss_curve=[]; val_curve=[]

for ep in range(1, EPOCHS+1):
    model.train(); scale=min(1.0,ep/WARMUP_EPOCHS); total=0.; opt.zero_grad()
    do_debug = (ep%10==0 or ep==1); acc_terms=None

    for i, c in enumerate(train_campuses):
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            ew, edge_logits = model(c['node_feats'], c['sub_ei'])
            loss = loss_fn(ew, edge_logits, c, scale) / n
        if not torch.isfinite(loss): loss = loss*0.
        scaler.scale(loss).backward(); total += loss.item()*n

        if do_debug:
            with torch.no_grad():
                terms={k:float(v.detach().cpu()) for k,v in loss_terms(ew,edge_logits,c,scale).items()}
                if acc_terms is None: acc_terms=terms.copy()
                else:
                    for k in terms: acc_terms[k]+=terms[k]

        if (i+1)%ACCUM_STEPS==0 or (i+1)==n:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    sch.step(); loss_curve.append(total/n)

    model.eval()
    with torch.no_grad():
        val_total=0.
        for c in val_campuses:
            with torch.amp.autocast('cuda',enabled=torch.cuda.is_available()):
                ew,edge_logits=model(c['node_feats'],c['sub_ei'])
                val_total+=loss_fn(ew,edge_logits,c,scale).item()
        val_curve.append(val_total/len(val_campuses))

    payload = {'epoch':ep,'model':model.state_dict(),'loss':total/n,'val':val_curve[-1]}
    torch.save(payload, CKPT)
    if val_curve[-1] <= min(val_curve):
        torch.save(payload, CKPT_BEST)
    if ep % 50 == 0:
        torch.save(payload, os.path.join(OUT_DIR, f'{CODE_VERSION}_ep{ep:03d}.pth'))

    msg=f'[{ep:4d}/{EPOCHS}] loss={total/n:.4f}  val={val_curve[-1]:.4f}  lr={sch.get_last_lr()[0]:.2e}'
    if acc_terms is not None:
        mean_t={k:v/n for k,v in acc_terms.items()}
        msg+='  | '+' '.join(f'{k}={v:.3f}' for k,v in mean_t.items())
    print(msg)

print(f'완료: {CKPT}')

In [ ]:
# ── 체크포인트 로드 + 손실 곡선 ─────────────────────────────────────────────
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=DEVICE)
    model.load_state_dict(ck['model'] if isinstance(ck, dict) else ck)
    print(f'체크포인트 로드: epoch={ck.get("epoch","?") if isinstance(ck,dict) else "?"}')

if loss_curve:
    plt.figure(figsize=(9,3))
    plt.plot(loss_curve, label='train')
    if val_curve: plt.plot(val_curve, label='val')
    plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'v28a_subgraph_mlp_loss_curve.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
# ── 시각화 헬퍼 ──────────────────────────────────────────────────────────────
def _prune(selected, terminal_sub, n_free, sub_ei_np):
    selected = selected.copy()
    term_set = set(int(t) for t in terminal_sub)
    src, dst = sub_ei_np[0], sub_ei_np[1]
    for _ in range(PRUNE_ITERS):
        deg = np.zeros(n_free, dtype=np.int32)
        for i, keep in enumerate(selected):
            if not keep: continue
            deg[src[i]] += 1; deg[dst[i]] += 1
        rm = np.zeros_like(selected, dtype=bool); changed = False
        for i, keep in enumerate(selected):
            if not keep: continue
            s, d = src[i], dst[i]
            if (deg[s] <= 1 and s not in term_set) or (deg[d] <= 1 and d not in term_set):
                rm[i] = True; changed = True
        selected[rm] = False
        if not changed: break
    return selected

def _largest_cc(selected, n_free, sub_ei_np):
    src, dst = sub_ei_np[0], sub_ei_np[1]
    adj = {}
    for i, keep in enumerate(selected):
        if not keep: continue
        adj.setdefault(src[i], []).append(dst[i])
        adj.setdefault(dst[i], []).append(src[i])
    if not adj: return selected
    seen = [False] * n_free; comps = []
    for v in adj:
        if seen[v]: continue
        stack = [v]; seen[v] = True; comp = []
        while stack:
            u = stack.pop(); comp.append(u)
            for nb in adj.get(u, []):
                if not seen[nb]: seen[nb] = True; stack.append(nb)
        comps.append(comp)
    keep_nodes = set(max(comps, key=len))
    out = selected.copy()
    for i, keep in enumerate(selected):
        if not keep: continue
        if src[i] not in keep_nodes or dst[i] not in keep_nodes: out[i] = False
    return out


_LABEL_BOX = dict(boxstyle='round,pad=0.15', facecolor='#0d1117', alpha=0.65, edgecolor='none')

def plot_campus(c, show_gt=True, use_prune=True):
    from matplotlib.lines import Line2D
    model.eval()
    with torch.no_grad():
        ew, _ = model(c['node_feats'], c['sub_ei'])
    ew_np     = ew.cpu().numpy()
    sub_ei_np = c['sub_ei'].cpu().numpy()
    fp        = c['free_pixels']
    W, H      = c['W'], c['H']

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')

    # 1. GT 엣지 (빨간, 맨 밑)
    if show_gt and c['gt_edge_mask'] is not None and c['gt_edge_mask'].sum() > 0:
        gt_em = c['gt_edge_mask'].cpu().numpy().astype(bool)
        for i, keep in enumerate(gt_em):
            if not keep: continue
            s_px = fp[sub_ei_np[0, i]]; d_px = fp[sub_ei_np[1, i]]
            y1, x1 = divmod(s_px, RES); y2, x2 = divmod(d_px, RES)
            ax.plot([x1, x2], [y1, y2], color='#ff6b6b', alpha=0.4, lw=1.5, zorder=2)

    # 2. 예측 엣지 (노란) — 건물보다 먼저 그려서 건물 아래로
    selected = ew_np > EDGE_THR
    if use_prune:
        selected = _largest_cc(selected, c['n_free'], sub_ei_np)
        selected = _prune(selected, c['terminal_sub'], c['n_free'], sub_ei_np)

    mxw = max(ew_np[selected].max() if selected.any() else EDGE_THR, EDGE_THR + 1e-8)
    for i, keep in enumerate(selected):
        if not keep: continue
        w = ew_np[i]
        s_px = fp[sub_ei_np[0, i]]; d_px = fp[sub_ei_np[1, i]]
        y1, x1 = divmod(s_px, RES); y2, x2 = divmod(d_px, RES)
        ax.plot([x1, x2], [y1, y2], color='#ffd32a', alpha=0.92,
                lw=max(1.2, (w-EDGE_THR)/(mxw-EDGE_THR+1e-8)*5+1),
                solid_capstyle='round', zorder=3)

    # 3. 건물 폴리곤 + ID — 엣지 위에 덮음
    for bid, pts in c['poly'].items():
        sc = [(p[0]*RES/W, p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc, closed=True,
                     facecolor='#2d3436', edgecolor='#00cec9', lw=0.8,
                     alpha=0.9, zorder=4))
        cx_b = np.mean([p[0] for p in sc])
        cy_b = np.mean([p[1] for p in sc])
        ax.text(cx_b, cy_b, str(bid), color='#dfe6e9',
                ha='center', va='center', fontsize=6.5, fontweight='bold',
                bbox=_LABEL_BOX, zorder=7)

    # 4. 클러스터·게이트 노드
    for ni, node in enumerate(c['cluster_nodes']):
        y, x = divmod(node, RES)
        ax.scatter(x, y, s=90, c='#74b9ff', edgecolors='white', lw=0.8, zorder=10)
        ax.text(x, y-2.8, f'C{ni}', color='#74b9ff',
                ha='center', fontsize=7, fontweight='bold', bbox=_LABEL_BOX, zorder=11)

    for gi, node in enumerate(c['gate_nodes']):
        y, x = divmod(node, RES)
        ax.scatter(x, y, s=110, c='#fdcb6e', marker='D', edgecolors='white', lw=0.8, zorder=10)
        ax.text(x, y-3.0, f'G{gi}', color='#fdcb6e',
                ha='center', fontsize=7, fontweight='bold', bbox=_LABEL_BOX, zorder=11)

    legend_elems = [
        mpatches.Patch(facecolor='#2d3436', edgecolor='#00cec9', label='Building'),
        Line2D([0],[0], color='#ffd32a', lw=2, label='Predicted road'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#74b9ff',
               markersize=7, label='Cluster node', linestyle='None'),
        Line2D([0],[0], marker='D', color='w', markerfacecolor='#fdcb6e',
               markersize=7, label='Gate node', linestyle='None'),
    ]
    if show_gt:
        legend_elems.insert(1, Line2D([0],[0], color='#ff6b6b', lw=2, alpha=0.5, label='GT road'))
    ax.legend(handles=legend_elems, loc='lower right', fontsize=8,
              facecolor='#1e272e', edgecolor='#636e72', labelcolor='white')

    n_ge   = int(c['gt_edge_mask'].sum()) if c['gt_edge_mask'] is not None else 0
    n_pred = int(selected.sum())
    split  = 'val' if c['slug'] in val_slugs else ('train' if c['gt_source']=='osm' else 'test')
    title_gt = f'  gt_edges={n_ge}' if show_gt else ''
    ax.set_xlim(0, RES); ax.set_ylim(RES, 0); ax.axis('off')
    ax.set_title(
        f'V28 · {c["name"].replace("_"," ").title()}  [{split}]\n'
        f'n_free={c["n_free"]}{title_gt}  pred_edges={n_pred}  '
        f'clusters={len(c["cluster_nodes"])}  gates={len(c["gate_nodes"])}',
        color='white', fontsize=11, pad=10
    )
    plt.tight_layout()
    save_path = os.path.join(OUT_DIR, f'{CODE_VERSION}_{c["name"]}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show(); plt.close()
    return save_path

print('plot_campus 정의 완료')

In [ ]:
# ── 테스트셋 전체 저장 ────────────────────────────────────────────────────────
print(f'테스트 캠퍼스 {len(test_campuses)}개 저장 시작...')
saved = []
for c in test_campuses:
    p = plot_campus(c, show_gt=False)
    saved.append(p)
print(f'\n저장 완료 ({len(saved)}개) → {OUT_DIR}/')